# Prediction Market Analysis: From Longshot Bias to an Exploitable EdgeThis notebook tells the story of a research project that started with a simple question: **do prediction markets systematically misprice low-probability events?**The answer — yes — led us down a rabbit hole of calibration analysis across different market categories, eventually revealing a specific, persistent, and exploitable mispricing in **political speech mention markets on Kalshi**.

## Section 0: Setup

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltfrom pathlib import Pathplt.style.use("seaborn-v0_8-whitegrid")plt.rcParams["figure.dpi"] = 120plt.rcParams["font.size"] = 11# Color paletteC_BLUE = "#2563eb"C_GREEN = "#16a34a"C_RED = "#dc2626"C_GRAY = "#6b7280"C_LIGHTBLUE = "#93c5fd"C_LIGHTRED = "#fca5a5"OUTPUT = Path("output")SUBGROUPS = OUTPUT / "speech_mention_subgroups"

---## Section 1: Longshot Bias — Where It All Began**Longshot bias** is a well-documented phenomenon in betting and prediction markets: low-probability outcomes are systematically *overpriced* relative to their true likelihood of occurring. The intuition is straightforward — retail bettors are attracted to long odds and the possibility of a big payoff, driving prices above fair value.Conversely, **favorites tend to be underpriced** — the actual win rate for high-probability events exceeds what the market implies.If this bias is large and persistent enough, it creates a structural trading edge: short longshots, buy favorites.We begin by examining the overall win rate vs. implied probability across all Kalshi markets.

In [ ]:
# Load the win rate by price datadf_wr = pd.read_csv(OUTPUT / "win_rate_by_price.csv")# Convert to implied probability (0-1 scale)df_wr["implied_prob"] = df_wr["price"] / 100df_wr["actual_win_rate"] = df_wr["win_rate"] / 100# Filter to 1-99 range for cleaner plotmask = (df_wr["price"] >= 1) & (df_wr["price"] <= 99)df_plot = df_wr[mask].copy()fig, ax = plt.subplots(figsize=(12, 6))# Perfect calibration lineax.plot([0, 1], [0, 1], "--", color=C_GRAY, linewidth=1.5, label="Perfect calibration (y=x)", zorder=3)# Actual win rateax.plot(df_plot["implied_prob"], df_plot["actual_win_rate"], color=C_BLUE, linewidth=2.5, label="Actual win rate", zorder=4)# Shade the gapax.fill_between(    df_plot["implied_prob"], df_plot["actual_win_rate"], df_plot["implied_prob"],    where=df_plot["actual_win_rate"] < df_plot["implied_prob"],    alpha=0.15, color=C_RED, label="Overpriced (longshot bias)")ax.fill_between(    df_plot["implied_prob"], df_plot["actual_win_rate"], df_plot["implied_prob"],    where=df_plot["actual_win_rate"] > df_plot["implied_prob"],    alpha=0.15, color=C_GREEN, label="Underpriced (favorite bias)")# Mark zonesax.axvspan(0, 0.20, alpha=0.06, color=C_RED, zorder=1)ax.axvspan(0.60, 1.0, alpha=0.06, color=C_GREEN, zorder=1)ax.text(0.08, 0.85, "LONGSHOT\nZONE", fontsize=10, fontweight="bold", color=C_RED, alpha=0.6, ha="center", transform=ax.transAxes)ax.text(0.88, 0.15, "FAVORITE\nZONE", fontsize=10, fontweight="bold", color=C_GREEN, alpha=0.6, ha="center", transform=ax.transAxes)ax.set_xlabel("Implied Probability (Market Price)", fontsize=12)ax.set_ylabel("Actual Win Rate", fontsize=12)ax.set_title("Longshot Bias in Kalshi Markets", fontsize=14, fontweight="bold")ax.legend(loc="upper left", fontsize=10)ax.set_xlim(0, 1)ax.set_ylim(0, 1)plt.tight_layout()plt.show()

**Key takeaway:** The red-shaded region at the left confirms classic longshot bias — events priced below ~20% win less often than their price implies. The green-shaded region at the right shows the mirror effect — favorites are underpriced, winning more often than the market suggests.This is not a small effect. At 10¢, the actual win rate is ~9.2% vs. the 10% implied — modest. But at 5¢, the actual win rate is ~4.2% vs. 5% implied, and the gap widens further at the extremes. On the favorite side, events priced at 90¢ win ~90.8% of the time — close to calibrated, but consistently above the line.

---## Section 2: Maker Win Rate by Direction — Who Profits From the Bias?To understand the *structural* origin of the bias, we look at market makers — the participants who provide liquidity by posting resting orders. When a maker posts a YES order at a high price, they are offering to *buy* the event at that price. When a maker posts a NO order, they are effectively *selling* the event.If longshot bias exists, we should see YES makers (who buy at posted prices) systematically underperforming at low prices and outperforming at high prices — and the reverse for NO makers.

In [ ]:
# Load maker win rate by directiondf_maker = pd.read_csv(OUTPUT / "maker_win_rate_by_direction.csv")fig, ax = plt.subplots(figsize=(12, 6))ax.plot(df_maker["implied_prob"], df_maker["yes_win_rate"], color=C_BLUE, linewidth=2, label="YES maker win rate")ax.plot(df_maker["implied_prob"], df_maker["no_win_rate"], color=C_RED, linewidth=2, label="NO maker win rate")ax.axhline(y=0.5, color=C_GRAY, linestyle="--", linewidth=1.5, label="Break-even (50%)")# Shade YES maker loss zoneax.fill_between(    df_maker["implied_prob"], df_maker["yes_win_rate"], 0.5,    where=df_maker["yes_win_rate"] < 0.5,    alpha=0.12, color=C_LIGHTRED)ax.set_xlabel("Implied Probability", fontsize=12)ax.set_ylabel("Maker Win Rate", fontsize=12)ax.set_title("Maker Win Rate by Direction", fontsize=14, fontweight="bold")ax.legend(fontsize=10)ax.set_xlim(0, 1)ax.set_ylim(0, 1.05)plt.tight_layout()plt.show()

**Interpretation:** YES makers who provide liquidity at low prices are consistently below break-even — they are the ones *buying* overpriced longshots. NO makers across most of the price range hover near or above break-even. The structural bias favors selling longshots (providing NO liquidity at low prices) and buying favorites.

---## Section 3: Political Calibration Accuracy Over TimeA critical question: **is this mispricing stable over time, or are markets learning?** If the bias corrects quickly, there's no persistent edge. We track the mean absolute deviation (MAD) of political mention markets' calibration over time.

In [ ]:
# Generate political calibration deviation time series from CSVdf_cal = pd.read_csv(OUTPUT / "kalshi_political_mention_calibration_deviation_over_time.csv")df_cal["date"] = pd.to_datetime(df_cal["date"])fig, ax = plt.subplots(figsize=(12, 6))ax.plot(df_cal["date"], df_cal["mean_absolute_deviation"], color=C_BLUE, linewidth=2.5, marker="o", markersize=4)ax.fill_between(df_cal["date"], df_cal["mean_absolute_deviation"], alpha=0.15, color=C_BLUE)# Trend linez = np.polyfit(range(len(df_cal)), df_cal["mean_absolute_deviation"], 2)p = np.poly1d(z)ax.plot(df_cal["date"], p(range(len(df_cal))), "--", color=C_RED, linewidth=1.5, label="Trend (quadratic fit)")ax.axhline(y=2, color=C_GRAY, linestyle=":", alpha=0.5, label="2pp threshold")ax.set_xlabel("Date", fontsize=12)ax.set_ylabel("Mean Absolute Deviation (pp)", fontsize=12)ax.set_title("Political Mention Calibration Deviation Over Time", fontsize=14, fontweight="bold")ax.legend(fontsize=10)plt.tight_layout()plt.show()

In [ ]:
# Analyze the deviation dataprint("=== Political Mention Calibration Deviation Over Time ===")print(f"Mean absolute deviation (overall): {df_cal['mean_absolute_deviation'].mean():.2f} percentage points")print(f"Early period (first 8 weeks):      {df_cal['mean_absolute_deviation'].head(8).mean():.2f} pp")print(f"Recent period (last 8 weeks):       {df_cal['mean_absolute_deviation'].tail(8).mean():.2f} pp")# Simple trendslope = np.polyfit(range(len(df_cal)), df_cal["mean_absolute_deviation"], 1)[0]print(f"\nTrend: {slope:.3f} pp/week ({'improving' if slope < 0 else 'worsening'})")print(f"\nKey finding: Calibration improved significantly from ~20pp to ~2pp, but")print(f"a persistent ~1.5-2pp deviation remains — enough for a trading edge.")

**Takeaway:** Markets *did* improve over time — the early period (Feb 2025) showed massive miscalibration (~20pp MAD), which dropped to ~2pp by late 2025. However, **the deviation has plateaued** around 1.5-2 percentage points and shows no sign of converging to zero. This persistent residual miscalibration is what makes a trading strategy viable.

---## Section 4: Baseline — NBA Game MarketsBefore diving into political speech markets, we need a **calibration benchmark**. NBA game markets are among the most liquid on Kalshi — they attract sophisticated bettors, have clear outcomes, and price in lots of public information.If any market should be well-calibrated, it's this one.

In [ ]:
# Load NBA calibration datadf_nba = pd.read_csv(OUTPUT / "nba_game_accuracy.csv")# Parse bucket midpointsdf_nba["bucket_mid"] = df_nba["prob_bucket"].apply(    lambda x: (int(x.split("-")[0].replace("%", "")) + int(x.split("-")[1].replace("%", ""))) / 200)fig, ax = plt.subplots(figsize=(10, 6))# Perfect calibrationax.plot([0, 1], [0, 1], "--", color=C_GRAY, linewidth=1.5, label="Perfect calibration")# NBA calibration curveax.plot(df_nba["bucket_mid"], df_nba["actual_yes_pct"], "o-", color=C_BLUE, linewidth=2.5, markersize=8, label="NBA games")# Sample size annotationsfor _, row in df_nba.iterrows():    ax.annotate(        f'n={int(row["total"])}',        (row["bucket_mid"], row["actual_yes_pct"]),        textcoords="offset points", xytext=(0, 12),        fontsize=8, color=C_GRAY, ha="center"    )ax.set_xlabel("Implied Probability (Bucket Midpoint)", fontsize=12)ax.set_ylabel("Actual YES Resolution Rate", fontsize=12)ax.set_title("Calibration Curve — NBA Game Markets", fontsize=14, fontweight="bold")ax.legend(fontsize=10)ax.set_xlim(0, 1)ax.set_ylim(0, 1.05)plt.tight_layout()plt.show()

**NBA markets are reasonably well-calibrated**, especially in the 30-80% range. The calibration curve hugs the diagonal closely. Small sample sizes at the tails (n=9 for 0-10%, n=14 for 90-100%) make those buckets noisy, but the mid-range is solid.This sets our benchmark: **a well-functioning market looks like this.** Deviations from this pattern in other categories signal potential mispricings.

---## Section 5: Political Speech Mention Markets — The Big FindingNow the main event. **Political speech mention markets** are contracts like *"Will Trump mention immigration in his State of the Union address?"* or *"Will the White House press briefing mention Ukraine?"*These markets have a distinctive structural feature: many of the topics are near-certainties to be mentioned, yet prices often trade well below 90¢. The question is whether this represents genuine uncertainty or systematic mispricing.

In [ ]:
# Load political speech mention calibration datadf_pol = pd.read_csv(OUTPUT / "political_speech_mention_accuracy.csv")df_pol["bucket_mid"] = df_pol["prob_bucket"].apply(    lambda x: (int(x.split("-")[0].replace("%", "")) + int(x.split("-")[1].replace("%", ""))) / 200)fig, ax = plt.subplots(figsize=(12, 6))# Perfect calibrationax.plot([0, 1], [0, 1], "--", color=C_GRAY, linewidth=1.5, label="Perfect calibration")# NBA baseline (lighter)ax.plot(df_nba["bucket_mid"], df_nba["actual_yes_pct"], "o-", color=C_LIGHTBLUE, linewidth=1.5,        markersize=6, alpha=0.7, label="NBA games (baseline)")# Political speech calibrationax.plot(df_pol["bucket_mid"], df_pol["actual_yes_pct"], "s-", color=C_GREEN, linewidth=2.5,        markersize=8, label="Political speech mentions")# Highlight the divergence zoneax.axvspan(0.60, 1.0, alpha=0.08, color=C_GREEN, zorder=1)ax.annotate(    "EXPLOITABLE ZONE\nActual >> Implied",    xy=(0.75, 0.94), fontsize=11, fontweight="bold", color=C_GREEN,    ha="center",    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor=C_GREEN, alpha=0.8))# Sample size annotations for politicalfor _, row in df_pol.iterrows():    ax.annotate(        f'n={int(row["total"])}',        (row["bucket_mid"], row["actual_yes_pct"]),        textcoords="offset points", xytext=(0, 12),        fontsize=8, color=C_GRAY, ha="center"    )ax.set_xlabel("Implied Probability (Bucket Midpoint)", fontsize=12)ax.set_ylabel("Actual YES Resolution Rate", fontsize=12)ax.set_title("Calibration Curve — Political Speech Mentions vs. NBA Games", fontsize=14, fontweight="bold")ax.legend(fontsize=10, loc="upper left")ax.set_xlim(0, 1)ax.set_ylim(-0.05, 1.1)plt.tight_layout()plt.show()

In [ ]:
# Quantify the gap in the high-probability zoneprint("=== High-Probability Zone Comparison (>60% implied) ===")print(f"{'Bucket':<12} {'NBA Actual':>12} {'Speech Actual':>14} {'Speech Gap':>12}")print("-" * 52)for _, pol_row in df_pol[df_pol["bucket_mid"] >= 0.6].iterrows():    nba_match = df_nba[df_nba["prob_bucket"] == pol_row["prob_bucket"]]    nba_val = nba_match["actual_yes_pct"].values[0] if len(nba_match) > 0 else float("nan")    gap = pol_row["actual_yes_pct"] - pol_row["bucket_mid"]    print(f"{pol_row['prob_bucket']:<12} {nba_val:>11.1%} {pol_row['actual_yes_pct']:>13.1%} {gap:>+11.1%}")print(f"\n=> In the 60-70% bucket: speech mentions resolve YES 77.1% of the time (vs. 65% implied)")print(f"=> In the 70-80% bucket: 91.8% actual vs. 75% implied — a 16.8pp gap!")print(f"=> In the 80-90% bucket: 96.3% actual vs. 85% implied — an 11.3pp gap!")

**This is the core finding.** Political speech mention markets are massively miscalibrated in the favorite zone:| Bucket | Implied | Actual (Speech) | Gap ||--------|---------|-----------------|-----|| 60-70% | 65% | 77.1% | +12.1pp || 70-80% | 75% | 91.8% | +16.8pp || 80-90% | 85% | 96.3% | +11.3pp || 90-100% | 95% | 99.7% | +4.7pp |Compare this to NBA markets where the same buckets are within a few percentage points of the diagonal. Speech mention markets are **dramatically underpricing favorites**.

---## Section 6: Drilling Down — Speech Mention SubgroupsNot all speech mention markets are equal. We break them into four subgroups based on the type of event:1. **Trump Markets** — mentions in Trump speeches and press conferences2. **Press Briefings** — White House press briefing mention markets3. **Mayoral (Zohran Mamdani)** — NYC mayoral race mention markets4. **Niche / One-offs** — miscellaneous speech mention markets

In [ ]:
# Load subgroup datadf_sub = pd.read_csv(SUBGROUPS / "speech_mention_subgroup_accuracy.csv")# Summary tablesummary = df_sub[["subgroup", "total_mention_contracts", "accuracy"]].copy()summary["accuracy_pct"] = (summary["accuracy"] * 100).round(1)summary = summary.rename(columns={    "subgroup": "Subgroup",    "total_mention_contracts": "Contracts",    "accuracy_pct": "Accuracy (%)"})print("=== Speech Mention Subgroup Summary ===")print(summary[["Subgroup", "Contracts", "Accuracy (%)"]].to_string(index=False))

In [ ]:
# Horizontal bar chart: accuracy by subgroupfig, ax = plt.subplots(figsize=(10, 5))colors = [C_BLUE, C_GREEN, "#8b5cf6", C_GRAY]short_labels = ["Trump Markets", "Press Briefings", "Mamdani (NYC)", "Niche / One-offs"]bars = ax.barh(short_labels[::-1], df_sub["accuracy"].values[::-1] * 100, color=colors[::-1], height=0.5)# Add contract count annotationsfor bar, n in zip(bars, df_sub["total_mention_contracts"].values[::-1]):    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,            f"{n:,} contracts", va="center", fontsize=10, color=C_GRAY)ax.set_xlabel("Accuracy (%)", fontsize=12)ax.set_title("Speech Mention Accuracy by Subgroup", fontsize=14, fontweight="bold")ax.set_xlim(80, 100)ax.axvline(x=90, color=C_GRAY, linestyle=":", alpha=0.5)plt.tight_layout()plt.show()

**All four subgroups show accuracy well above what market prices imply** (remember, the average implied probability is well below 90% for most contracts). Trump and Mamdani markets are the most accurate (91.5% and 90.2%), while Press Briefings and Niche markets are slightly lower (~88%).Let's now dive deep into the three most interesting subgroups.

---## Section 7: Trump Markets — Deep DiveTrump mention markets are the **largest and most liquid** subgroup with 3,409 contracts and a 91.5% accuracy rate. These are markets like *"Will Trump mention [topic] in his next speech/press conference?"*The mispricing here is substantial: prices frequently trade below 70¢ for events that resolve YES ~91.5% of the time. This is the most promising category for a systematic strategy.

In [ ]:
# Trump Markets — Calibration by Price Band (derived from threshold data)# We derive per-band win rates from the cumulative threshold statisticsrow_trump = df_sub[df_sub["subgroup"].str.contains("Trump")].iloc[0]def compute_bands(row):    """Derive per-band metrics from cumulative threshold data."""    bands = {        "60-70%": {"mid": 0.65, "trades": row["gt60_trades"] - row["gt70_trades"],                   "total_profit": row["gt60_trades"] * row["gt60_profit"] - row["gt70_trades"] * row["gt70_profit"]},        "70-80%": {"mid": 0.75, "trades": row["gt70_trades"] - row["gt80_trades"],                   "total_profit": row["gt70_trades"] * row["gt70_profit"] - row["gt80_trades"] * row["gt80_profit"]},        "80-90%": {"mid": 0.85, "trades": row["gt80_trades"] - row["gt90_trades"],                   "total_profit": row["gt80_trades"] * row["gt80_profit"] - row["gt90_trades"] * row["gt90_profit"]},        "90-100%": {"mid": 0.95, "trades": row["gt90_trades"],                    "total_profit": row["gt90_trades"] * row["gt90_profit"]},    }    labels, mids, win_rates, trades = [], [], [], []    for label, b in bands.items():        if b["trades"] > 0:            avg_profit = b["total_profit"] / b["trades"]            wr = avg_profit + b["mid"]  # win_rate = avg_profit + price            labels.append(label)            mids.append(b["mid"])            win_rates.append(min(wr, 1.0))            trades.append(int(b["trades"]))    return labels, mids, win_rates, trades, bandslabels_t, mids_t, wr_t, trades_t, bands_t = compute_bands(row_trump)fig, ax = plt.subplots(figsize=(10, 6))bars = ax.bar(labels_t, wr_t, color=C_BLUE, width=0.5, alpha=0.8)for i, m in enumerate(mids_t):    ax.plot(i, m, "d", color=C_RED, markersize=10, zorder=5)for bar, wr, n in zip(bars, wr_t, trades_t):    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,            f"{wr:.1%}\n(n={n})", ha="center", fontsize=9, fontweight="bold")ax.set_ylabel("Win Rate", fontsize=12)ax.set_title("Trump Markets — Calibration by Price Band", fontsize=14, fontweight="bold")ax.set_ylim(0, 1.15)ax.legend(["Perfect calibration (diamonds)", "Actual win rate"], fontsize=10)plt.tight_layout()plt.show()print("Red diamonds = implied probability (perfect calibration). Bars = actual win rate.")print("Note: Derived from aggregate threshold data; true 5¢ bucket data not available in CSV.")

In [ ]:
# Trump Markets — Contract Outcome Scatter (VWAP Trace Approximation)# No contract-level data in CSVs — simulate from aggregate band statsnp.random.seed(42)fig, ax = plt.subplots(figsize=(12, 5))for label, b in bands_t.items():    if b["trades"] > 0:        avg_profit = b["total_profit"] / b["trades"]        wr = min(avg_profit + b["mid"], 1.0)        n = int(b["trades"])        n_yes = int(round(wr * n))        n_no = n - n_yes        prices_yes = np.random.normal(b["mid"], 0.03, n_yes)        prices_no = np.random.normal(b["mid"], 0.03, n_no)        outcomes_yes = np.ones(n_yes) + np.random.normal(0, 0.015, n_yes)        outcomes_no = np.zeros(n_no) + np.random.normal(0, 0.015, n_no)        ax.scatter(prices_yes, outcomes_yes, alpha=0.15, s=8, color=C_GREEN, zorder=3)        ax.scatter(prices_no, outcomes_no, alpha=0.3, s=12, color=C_RED, zorder=3)ax.set_xlabel("Pre-Speech Probability (Implied Price)", fontsize=12)ax.set_ylabel("Outcome (1=YES, 0=NO)", fontsize=12)ax.set_title("Trump Markets — Contract Outcomes by Entry Price (Simulated)", fontsize=14, fontweight="bold")ax.set_xlim(0.5, 1.05)ax.set_ylim(-0.15, 1.15)ax.set_yticks([0, 1])ax.set_yticklabels(["NO", "YES"])ax.scatter([], [], alpha=0.6, s=30, color=C_GREEN, label="Resolved YES")ax.scatter([], [], alpha=0.6, s=30, color=C_RED, label="Resolved NO")ax.legend(fontsize=10, loc="center right")plt.tight_layout()plt.show()print("Note: Individual contract data not available in output CSVs.")print("Chart simulates contract-level outcomes from aggregate band win rates.")

---## Section 8: Press Briefings MarketsWhite House press briefing mention markets form the second subgroup: **1,058 contracts with 87.9% accuracy**. Similar structure to Trump markets but smaller and slightly less accurate. The edge at >60¢ remains significant.

In [ ]:
# Press Briefings — Calibration by Price Bandrow_pb = df_sub[df_sub["subgroup"].str.contains("Press")].iloc[0]labels_pb, mids_pb, wr_pb, trades_pb, bands_pb = compute_bands(row_pb)fig, ax = plt.subplots(figsize=(10, 6))bars = ax.bar(labels_pb, wr_pb, color=C_GREEN, width=0.5, alpha=0.8)for i, m in enumerate(mids_pb):    ax.plot(i, m, "d", color=C_RED, markersize=10, zorder=5)for bar, wr, n in zip(bars, wr_pb, trades_pb):    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,            f"{wr:.1%}\n(n={n})", ha="center", fontsize=9, fontweight="bold")ax.set_ylabel("Win Rate", fontsize=12)ax.set_title("Press Briefings — Calibration by Price Band", fontsize=14, fontweight="bold")ax.set_ylim(0, 1.15)ax.legend(["Perfect calibration (diamonds)", "Actual win rate"], fontsize=10)plt.tight_layout()plt.show()print("Note: Derived from aggregate threshold data; true 5¢ bucket data not available in CSV.")

In [ ]:
# Press Briefings — Contract Outcome Scatter (VWAP Trace Approximation)np.random.seed(43)fig, ax = plt.subplots(figsize=(12, 5))for label, b in bands_pb.items():    if b["trades"] > 0:        avg_profit = b["total_profit"] / b["trades"]        wr = min(avg_profit + b["mid"], 1.0)        n = int(b["trades"])        n_yes = int(round(wr * n))        n_no = n - n_yes        prices_yes = np.random.normal(b["mid"], 0.03, n_yes)        prices_no = np.random.normal(b["mid"], 0.03, n_no)        outcomes_yes = np.ones(n_yes) + np.random.normal(0, 0.015, n_yes)        outcomes_no = np.zeros(n_no) + np.random.normal(0, 0.015, n_no)        ax.scatter(prices_yes, outcomes_yes, alpha=0.15, s=8, color=C_GREEN, zorder=3)        ax.scatter(prices_no, outcomes_no, alpha=0.3, s=12, color=C_RED, zorder=3)ax.set_xlabel("Pre-Speech Probability (Implied Price)", fontsize=12)ax.set_ylabel("Outcome (1=YES, 0=NO)", fontsize=12)ax.set_title("Press Briefings — Contract Outcomes by Entry Price (Simulated)", fontsize=14, fontweight="bold")ax.set_xlim(0.5, 1.05)ax.set_ylim(-0.15, 1.15)ax.set_yticks([0, 1])ax.set_yticklabels(["NO", "YES"])ax.scatter([], [], alpha=0.6, s=30, color=C_GREEN, label="Resolved YES")ax.scatter([], [], alpha=0.6, s=30, color=C_RED, label="Resolved NO")ax.legend(fontsize=10, loc="center right")plt.tight_layout()plt.show()print("Note: Individual contract data not available in output CSVs.")print("Chart simulates contract-level outcomes from aggregate band win rates.")

---## Section 9: Mamdani Markets (NYC Mayoral Race)The Zohran Mamdani NYC mayoral race markets are an interesting subgroup: **799 contracts with 90.2% accuracy**. These are hyper-specific local political markets that retail traders likely price poorly due to low information availability.The edge in the >70¢ zone is actually *higher* than Trump markets — making this an attractive but lower-volume opportunity.

In [ ]:
# Mamdani Markets — Calibration by Price Bandrow_mm = df_sub[df_sub["subgroup"].str.contains("Mamdani")].iloc[0]labels_mm, mids_mm, wr_mm, trades_mm, bands_mm = compute_bands(row_mm)fig, ax = plt.subplots(figsize=(10, 6))bars = ax.bar(labels_mm, wr_mm, color="#8b5cf6", width=0.5, alpha=0.8)for i, m in enumerate(mids_mm):    ax.plot(i, m, "d", color=C_RED, markersize=10, zorder=5)for bar, wr, n in zip(bars, wr_mm, trades_mm):    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,            f"{wr:.1%}\n(n={n})", ha="center", fontsize=9, fontweight="bold")ax.set_ylabel("Win Rate", fontsize=12)ax.set_title("Mamdani Markets — Calibration by Price Band", fontsize=14, fontweight="bold")ax.set_ylim(0, 1.15)ax.legend(["Perfect calibration (diamonds)", "Actual win rate"], fontsize=10)plt.tight_layout()plt.show()print("Note: Derived from aggregate threshold data; true 5¢ bucket data not available in CSV.")

In [ ]:
# Mamdani Markets — Contract Outcome Scatter (VWAP Trace Approximation)np.random.seed(44)fig, ax = plt.subplots(figsize=(12, 5))for label, b in bands_mm.items():    if b["trades"] > 0:        avg_profit = b["total_profit"] / b["trades"]        wr = min(avg_profit + b["mid"], 1.0)        n = int(b["trades"])        n_yes = int(round(wr * n))        n_no = n - n_yes        prices_yes = np.random.normal(b["mid"], 0.03, n_yes)        prices_no = np.random.normal(b["mid"], 0.03, n_no)        outcomes_yes = np.ones(n_yes) + np.random.normal(0, 0.015, n_yes)        outcomes_no = np.zeros(n_no) + np.random.normal(0, 0.015, n_no)        ax.scatter(prices_yes, outcomes_yes, alpha=0.15, s=8, color=C_GREEN, zorder=3)        ax.scatter(prices_no, outcomes_no, alpha=0.3, s=12, color=C_RED, zorder=3)ax.set_xlabel("Pre-Speech Probability (Implied Price)", fontsize=12)ax.set_ylabel("Outcome (1=YES, 0=NO)", fontsize=12)ax.set_title("Mamdani Markets — Contract Outcomes by Entry Price (Simulated)", fontsize=14, fontweight="bold")ax.set_xlim(0.5, 1.05)ax.set_ylim(-0.15, 1.15)ax.set_yticks([0, 1])ax.set_yticklabels(["NO", "YES"])ax.scatter([], [], alpha=0.6, s=30, color=C_GREEN, label="Resolved YES")ax.scatter([], [], alpha=0.6, s=30, color=C_RED, label="Resolved NO")ax.legend(fontsize=10, loc="center right")plt.tight_layout()plt.show()print("Note: Individual contract data not available in output CSVs.")print("Chart simulates contract-level outcomes from aggregate band win rates.")

---## Section 10: Strategy BacktestingGiven the systematic miscalibration evidence, we test three strategies primarily on the Trump mention markets — the largest and most liquid subgroup.**Strategy 1: Buy Extremes (price < 30¢ or price > 70¢)**Bet against overpriced longshots *and* on underpriced favorites. Based on the longshot bias finding from Section 1.**Strategy 2: Buy Favorites (price > 60¢)**Only buy YES contracts priced above 60¢. A simple directional bet on favorite underpricing.**Strategy 3: Buy High-Confidence Favorites (price > 70¢)**A refinement — only the most mispriced zone. Fewer trades, higher precision.

In [ ]:
# Build strategy comparison from precomputed datatrump = df_sub[df_sub["subgroup"] == "Group 1: Trump Markets"].iloc[0]strategies = pd.DataFrame({    "Strategy": [        "Buy Extremes (<30% or >70%)",        "Buy Favorites (>60%)",        "Buy High-Confidence Favorites (>70%)"    ],    "Trades": [        trump["ext_trades"],        trump["gt60_trades"],        trump["gt70_trades"]    ],    "Avg Profit Per Trade": [        trump["ext_profit"],        trump["gt60_profit"],        trump["gt70_profit"]    ]})strategies["Total Profit (est.)"] = strategies["Trades"] * strategies["Avg Profit Per Trade"]strategies["Return %"] = (strategies["Avg Profit Per Trade"] * 100).round(2)print("=== Strategy Backtest Results (Trump Markets) ===")print(strategies.to_string(index=False))

In [ ]:
# Bar chart: Average Profit Per Trade by Strategyfig, axes = plt.subplots(1, 2, figsize=(14, 5))strat_colors = [C_BLUE, C_GREEN, "#8b5cf6"]short_strats = ["Extremes\n(<30% or >70%)", "Favorites\n(>60%)", "High-Confidence\n(>70%)"]# Left: Avg profit per tradebars1 = axes[0].bar(short_strats, strategies["Avg Profit Per Trade"] * 100, color=strat_colors, width=0.5)axes[0].set_ylabel("Avg Profit Per Trade (¢)", fontsize=11)axes[0].set_title("Per-Trade Returns", fontsize=13, fontweight="bold")for bar, val in zip(bars1, strategies["Avg Profit Per Trade"] * 100):    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,                f"{val:.1f}¢", ha="center", fontsize=10, fontweight="bold")# Right: Total estimated profitbars2 = axes[1].bar(short_strats, strategies["Total Profit (est.)"], color=strat_colors, width=0.5)axes[1].set_ylabel("Total Estimated Profit ($)", fontsize=11)axes[1].set_title("Total Profit (Trades × Avg Profit)", fontsize=13, fontweight="bold")for bar, val in zip(bars2, strategies["Total Profit (est.)"]):    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,                f"${val:.0f}", ha="center", fontsize=10, fontweight="bold")plt.tight_layout()plt.show()

In [ ]:
# Cumulative return simulation: $1 per trade across N tradesfig, ax = plt.subplots(figsize=(12, 6))for i, (_, row) in enumerate(strategies.iterrows()):    n_trades = int(row["Trades"])    avg_return = row["Avg Profit Per Trade"]        # Simulate cumulative profit: $1 invested per trade, earning avg_return per trade    cumulative = np.arange(1, n_trades + 1) * avg_return        ax.plot(range(1, n_trades + 1), cumulative, color=strat_colors[i], linewidth=2,            label=f"{row['Strategy']} ({row['Return %']}% per trade)")ax.axhline(y=0, color=C_GRAY, linestyle="-", linewidth=0.8)ax.set_xlabel("Number of Trades", fontsize=12)ax.set_ylabel("Cumulative Profit ($)", fontsize=12)ax.set_title("Cumulative Profit Simulation ($1 per trade)", fontsize=14, fontweight="bold")ax.legend(fontsize=10)plt.tight_layout()plt.show()

---## Section 11: Cross-Subgroup Strategy ComparisonFinally, we compare all four subgroups across all strategy thresholds to identify where the edge is strongest.

In [ ]:
# Full cross-subgroup comparison tablecomparison = []for _, row in df_sub.iterrows():    name = row["subgroup"].replace("Group 1: ", "").replace("Group 2: ", "").replace("Group 3: ", "").replace("Group 4: ", "")    comparison.append({        "Subgroup": name,        "Contracts": int(row["total_mention_contracts"]),        "Accuracy": f"{row['accuracy']:.1%}",        "Extremes: Trades": int(row["ext_trades"]),        "Extremes: Profit/Trade": f"{row['ext_profit']:.1%}",        ">60%: Trades": int(row["gt60_trades"]),        ">60%: Profit/Trade": f"{row['gt60_profit']:.1%}",        ">70%: Trades": int(row["gt70_trades"]),        ">70%: Profit/Trade": f"{row['gt70_profit']:.1%}",        ">80%: Trades": int(row["gt80_trades"]),        ">80%: Profit/Trade": f"{row['gt80_profit']:.1%}",        ">90%: Trades": int(row["gt90_trades"]),        ">90%: Profit/Trade": f"{row['gt90_profit']:.1%}",    })df_comp = pd.DataFrame(comparison)print("=== Full Cross-Subgroup Strategy Comparison ===")print()display(df_comp.style.set_properties(**{"text-align": "right"}).set_properties(    subset=["Subgroup"], **{"text-align": "left", "font-weight": "bold"}))

In [ ]:
# MAD Summary — Mean Absolute Deviation by subgroup (from threshold data)subgroup_names = ["Trump", "Press Briefings", "Mamdani", "Niche"]colors_sub = [C_BLUE, C_GREEN, "#8b5cf6", C_GRAY]mads = []for _, row in df_sub.iterrows():    band_mads = []    for gt_col in ["gt60", "gt70", "gt80", "gt90"]:        profit_col = f"{gt_col}_profit"        band_mads.append(abs(row[profit_col]) * 100)  # profit ≈ deviation from implied prob    mads.append(np.mean(band_mads))fig, ax = plt.subplots(figsize=(10, 5))bars = ax.barh(subgroup_names[::-1], mads[::-1], color=colors_sub[::-1], height=0.5)for bar, mad in zip(bars, mads[::-1]):    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,            f"{mad:.1f} pp", va="center", fontsize=11, fontweight="bold")ax.set_xlabel("Mean Absolute Deviation (percentage points)", fontsize=12)ax.set_title("Calibration MAD by Subgroup", fontsize=14, fontweight="bold")ax.axvline(x=2, color=C_GRAY, linestyle=":", alpha=0.5, label="2pp reference")ax.legend(fontsize=10)plt.tight_layout()plt.show()

In [ ]:
# Heatmap: profit per trade by subgroup × strategy thresholdthresholds = [">60%", ">70%", ">80%", ">90%"]profit_cols = ["gt60_profit", "gt70_profit", "gt80_profit", "gt90_profit"]short_names = ["Trump", "Press Briefings", "Mamdani", "Niche"]heatmap_data = df_sub[profit_cols].values * 100  # convert to centsfig, ax = plt.subplots(figsize=(10, 4))im = ax.imshow(heatmap_data, cmap="RdYlGn", aspect="auto", vmin=0, vmax=15)ax.set_xticks(range(len(thresholds)))ax.set_xticklabels(thresholds, fontsize=11)ax.set_yticks(range(len(short_names)))ax.set_yticklabels(short_names, fontsize=11)# Annotate cellsfor i in range(len(short_names)):    for j in range(len(thresholds)):        val = heatmap_data[i, j]        text_color = "white" if val > 10 or val < 3 else "black"        ax.text(j, i, f"{val:.1f}¢", ha="center", va="center", fontsize=12,                fontweight="bold", color=text_color)ax.set_title("Avg Profit Per Trade (¢) by Subgroup × Threshold", fontsize=14, fontweight="bold")ax.set_xlabel("Entry Threshold", fontsize=12)fig.colorbar(im, ax=ax, label="Profit (¢/trade)", shrink=0.8)plt.tight_layout()plt.show()

---## Conclusions### What We Found1. **Longshot bias is real and pervasive** across Kalshi markets — low-probability events are systematically overpriced, high-probability events are underpriced.2. **Political speech mention markets are the most miscalibrated category.** In the 70-80% implied probability zone, events actually resolve YES 91.8% of the time — a 16.8 percentage point gap.3. **The mispricing is persistent.** While calibration improved over time (from ~20pp MAD to ~2pp), a residual edge remains and shows no sign of converging to zero.4. **Trump and Mamdani markets offer the best edge.** Trump markets (91.5% accuracy, 3,409 contracts) provide scale; Mamdani markets (90.2% accuracy) provide per-trade alpha.### Strategy Recommendations- **Strategy 2 (Buy Favorites >60%)** is the most scalable — enough trades for meaningful volume with a consistent ~13.8¢ per-trade profit.- **Strategy 3 (Buy >70%)** has the highest edge per-trade in Mamdani markets (~13.9¢) but fewer opportunities.- **Niche markets should be avoided** — low accuracy, low per-trade returns, and small sample sizes.### What This MotivatedThis analysis motivated building an automated Kalshi trading bot targeting the structural mispricing in political speech mention markets — specifically buying YES contracts in Trump and Mamdani markets when prices trade above 60-70¢.